<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [29]:
import numpy as np
import matplotlib.pyplot as plt
from random import randint, choices, sample, random
import cv2
from abc import ABC, abstractmethod
from copy import deepcopy

Create generic Solution class as seen in Labs with methods fitness and random initial representation is repr is None

In [17]:
class Solution(ABC):
    def __init__(self, repr=None):
        if repr == None:
            repr = self.random_initial_representation()
        self.repr = repr

    def __repr__(self):
        return str(self.repr)
    
    @abstractmethod
    def fitness(self):
        pass

    @abstractmethod
    def random_initial_representation(self):
        pass

In [18]:
class VermeerSolution(Solution):
    def __init__(self, target_image, repr=None):
        """"Class with the attributes highlighted within the project description width, heigh and num_triangles"""
        self.target_image = target_image
        self.width = 300
        self.height = 400
        self.num_triangles = 100
        super().__init__(repr=repr)

    def random_initial_representation(self):
        """
        Generates random canvas of 100 triangles. Returns a numpy array of shape (100, 9)
        Columns: [x1, y1, x2, y2, x3, y3, r, g, b]
        """
        # Create empty array of zeros to hold 100 triangles
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)

        for i in range(self.num_triangles):
            # Generate random vertices (x, y) within 300x400 canvas
            x1, x2, x3 = [randint(0, self.width) for _ in range(3)]
            y1, y2, y3 = [randint(0, self.height) for _ in range(3)]

            # Generate random rgb colors
            r, g, b = [randint(0, 255) for _ in range(3)]

            # Store triangle in array
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]
        
        return random_repr
    
    def render_canvas(self):
        """
        Converts 100-triangle array into a 300x400 pixel image matrix
        """
        # Create a blank canvas again with 300 width, 400 height and 3 color chanels (r,g,b)
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        for gene in self.repr:
            # Extract the 3 (x,y) vertices and reshape them for OpenCV
            # Format: [[x1, y1], [x2, y2], [x3, y3]]
            pts = np.array([
                [gene[0], gene[1]],
                [gene[2], gene[3]],
                [gene[4], gene[5]]
            ], np.int32)

            pts = pts.reshape((-1, 1, 2))

            # Extract RGB 
            color = (int(gene[6]), int(gene[7]), int(gene[8]))
            cv2.fillPoly(canvas, [pts], color)
        return canvas
    

    def fitness(self):
        """
        Calculates RMSE between target image and generated image (Minimization problem)
        """
        generated_image = self.render_canvas()
        # Convert uint8 arrays to float32 before subtracting by zero or will cause Error
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)
        
        # Calculate pixel by pixel difference between generated image and target image
        diff = target_float - gen_float
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        return rmse


Now to test out if the initial class functions in producing an image and comparing it to the original painting

In [ ]:
"""#gonçalo_path = 'C:\CIfO\Project\data\girl_pearl_earing.png'

original_vermeer = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')

my_first_painting = VermeerSolution(target_image=original_vermeer)

score = round(my_first_painting.fitness(), 2)

print(f'The RMSE Score of random painting is: {score}')"""



The RMSE Score of random painting is: 90.41000366210938


Now I will create my GA class to perform optimization of my solution

**Notes**:
The probability of mutation should be 1 over the length of the representation (string)

In [ ]:
class VermeerGA:
    def __init__(self, target_image, pop_size=50, generations=1000, mutation_rate=0.05, tournament_size=3):
        self.target_image = target_image
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size
        
        # Initialize the population with random paintings
        self.population = [VermeerSolution(target_image=self.target_image) for _ in range(self.pop_size)]

    def tournament_selection(self):
        """Picks K random individuals, returns the one with the lowest RMSE."""
        tournament = sample(self.population, self.tournament_size)
        # Sort by the pre-calculated fitness_score attribute
        winner = sorted(tournament, key=lambda ind: ind.fitness_score)[0]
        return winner

    def crossover(self, parent1, parent2):
        """Single-point crossover splitting the 100 triangles at index 50."""
        child = VermeerSolution(target_image=self.target_image)
        # This is very wrong refer back to notes to change this aspect of the genetic algorithm (make this a random split points
        # for the length of the representation of the triangles)
        split_point = 50
        
        # Inherit half from parent 1, half from parent 2
        child.repr[:split_point] = parent1.repr[:split_point]
        child.repr[split_point:] = parent2.repr[split_point:]
        return child

    def mutate(self, child):
        """Randomly nudges vertices and colors of the triangles."""
        for i in range(child.num_triangles):
            if random() < self.mutation_rate:
                # Creep mutation on (X, Y) coordinates
                child.repr[i][0:6] += np.random.randint(-15, 16, 6)
                # Creep mutation on (R, G, B) colors
                child.repr[i][6:9] += np.random.randint(-20, 21, 3)
                
                # Clip values to ensure they stay within canvas and color bounds
                child.repr[i][0::2] = np.clip(child.repr[i][0::2], 0, child.width)
                child.repr[i][1::2] = np.clip(child.repr[i][1::2], 0, child.height)
                child.repr[i][6:9] = np.clip(child.repr[i][6:9], 0, 255)
        return child

    def run(self):
        """The main evolutionary loop."""
        print(f"Starting Evolution for {self.generations} generations...")
        
        # Track history for plotting in your final report!
        history = []

        for gen in range(self.generations):
            # 1. Evaluate the entire population
            for individual in self.population:
                # We save the score as an attribute so we don't recalculate it constantly
                individual.fitness_score = individual.fitness()
            
            # 2. Sort population from best (lowest RMSE) to worst
            self.population.sort(key=lambda ind: ind.fitness_score)
            best_current_solution = self.population[0]
            
            # Save the best score of this generation for your report
            history.append(best_current_solution.fitness_score)
            
            if gen % 10 == 0:
                print(f"Generation {gen} | Best RMSE: {best_current_solution.fitness_score:.2f}")
            
            # 3. Create the next generation
            next_generation = []
            
            # ELITISM: Keep the absolute best painting unchanged
            next_generation.append(deepcopy(best_current_solution))
            
            # 4. Breed the rest of the new generation
            while len(next_generation) < self.pop_size:
                # Select parents
                p1 = self.tournament_selection()
                p2 = self.tournament_selection()
                
                # Crossover
                child = self.crossover(p1, p2)
                
                # Mutate
                child = self.mutate(child)
                
                next_generation.append(child)
                
            # 5. Replace the old population with the new one
            self.population = next_generation
            
        print("Evolution Complete!")
        # Return the absolute best painting and the historical data
        return self.population[0], history

In [33]:
if __name__ == "__main__":
    # 1. Load the target image
    target = cv2.imread(r'C:\CIfO\Project\data\girl_pearl_earing.png')
    if target is None:
        raise FileNotFoundError("OpenCV could not find the image! Check the file name and folder path.")
    
    # 2. Initialize the Genetic Algorithm
    # Note: Start with a small population (e.g., 20) and low generations (e.g., 50) 
    # just to test that your code runs without crashing before doing a massive overnight run!
    ga = VermeerGA(target_image=target, pop_size=50, generations=1000, mutation_rate=0.05)
    
    # 3. Run the optimization!
    best_painting, fitness_history = ga.run()
    
    # 4. Render and view the final result
    final_image = best_painting.render_canvas()
    cv2.imshow("Best Vermeer Approximation", final_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    

Starting Evolution for 1000 generations...
Generation 0 | Best RMSE: 85.14
Generation 10 | Best RMSE: 80.54
Generation 20 | Best RMSE: 75.67
Generation 30 | Best RMSE: 70.94
Generation 40 | Best RMSE: 68.02
Generation 50 | Best RMSE: 65.94
Generation 60 | Best RMSE: 64.23
Generation 70 | Best RMSE: 62.12
Generation 80 | Best RMSE: 60.72
Generation 90 | Best RMSE: 59.24
Generation 100 | Best RMSE: 58.20
Generation 110 | Best RMSE: 57.23
Generation 120 | Best RMSE: 56.30
Generation 130 | Best RMSE: 54.92
Generation 140 | Best RMSE: 54.07
Generation 150 | Best RMSE: 53.66
Generation 160 | Best RMSE: 53.20
Generation 170 | Best RMSE: 52.50
Generation 180 | Best RMSE: 51.72
Generation 190 | Best RMSE: 51.00
Generation 200 | Best RMSE: 50.36
Generation 210 | Best RMSE: 50.04
Generation 220 | Best RMSE: 49.63
Generation 230 | Best RMSE: 49.46
Generation 240 | Best RMSE: 49.23
Generation 250 | Best RMSE: 48.85
Generation 260 | Best RMSE: 48.62
Generation 270 | Best RMSE: 48.39
Generation 280 |

In [ ]:
""""
The last Run performed with the following Parameters:
Population: 200
Nº Generations: 2000
RMSE @ Generation Nº 230: 46.18
Best RMSE: 34. 28 
"""